In [9]:
import pandas as pd
import numpy as np
print("Libraries loaded successfully!")

Libraries loaded successfully!


In [10]:
# === Load data and set IS_DAYTIME type =====
df = pd.read_csv('merged_cleaned_scaled.csv')
df['DATE_TIME'] = pd.to_datetime(df['DATE_TIME'])
df['IS_DAYTIME'] = df['IS_DAYTIME'].astype(bool)  # Ensure boolean type after loading from CSV
print("Loaded shape:", df.shape)

Loaded shape: (136476, 15)


In [11]:
# Each of the 44 inverters is its own independent time series,
# so we group by (PLANT_ID, SOURCE_KEY) before sorting by time
df = df.sort_values(['PLANT_ID', 'SOURCE_KEY', 'DATE_TIME']).reset_index(drop=True)

In [12]:
# Team decision: only 3 environmental inputs, AC_POWER excluded from inputs
feature_cols = ['IRRADIATION_scaled', 'AMBIENT_TEMPERATURE_scaled', 'MODULE_TEMPERATURE_scaled']
target_col = 'AC_POWER_scaled'

In [13]:
# Split by calendar time, not randomly, to avoid the model "seeing the future"
min_date, max_date = df['DATE_TIME'].min(), df['DATE_TIME'].max()
total_seconds = (max_date - min_date).total_seconds()
train_end = min_date + pd.Timedelta(seconds=total_seconds * 0.70)
val_end   = min_date + pd.Timedelta(seconds=total_seconds * 0.85)

print("Train ends:", train_end)
print("Validation ends:", val_end)

Train ends: 2020-06-07 19:01:29.999999999
Validation ends: 2020-06-12 21:23:15


In [14]:
# =====  Build windows while excluding nighttime targets =====
def build_windows(window_size):
    X_train, y_train = [], []
    X_val,   y_val   = [], []
    X_test,  y_test  = [], []

    for (plant, inverter), group in df.groupby(['PLANT_ID', 'SOURCE_KEY']):
        group = group.reset_index(drop=True)
        feats = group[feature_cols].values
        target = group[target_col].values
        times = pd.to_datetime(group['DATE_TIME'].values)
        is_daytime = group['IS_DAYTIME'].values  # Daytime flag

        is_continuous = (np.diff(times) == np.timedelta64(15, 'm'))

        n = len(group)
        for i in range(n - window_size):
            if not is_continuous[i:i + window_size].all():
                continue

            # Exclude windows where the target timestamp is nighttime
            # Nighttime targets are trivially zero and can inflate performance
            # and distort the anomaly detection threshold (mean/std of residuals)
            if not is_daytime[i + window_size]:
                continue

            X_seq = feats[i:i + window_size]
            y_next = target[i + window_size]
            t_target = times[i + window_size]

            if t_target < train_end:
                X_train.append(X_seq); y_train.append(y_next)
            elif t_target < val_end:
                X_val.append(X_seq); y_val.append(y_next)
            else:
                X_test.append(X_seq); y_test.append(y_next)

    return (np.array(X_train), np.array(y_train),
            np.array(X_val),   np.array(y_val),
            np.array(X_test),  np.array(y_test))

In [15]:
for window_size in [24, 32, 48]:
    X_train, y_train, X_val, y_val, X_test, y_test = build_windows(window_size)

    print(f"\n--- Window size {window_size} ---")
    print("Train:", X_train.shape, y_train.shape)
    print("Val:  ", X_val.shape, y_val.shape)
    print("Test: ", X_test.shape, y_test.shape)

    np.save(f'X_train_w{window_size}.npy', X_train)
    np.save(f'y_train_w{window_size}.npy', y_train)
    np.save(f'X_val_w{window_size}.npy',   X_val)
    np.save(f'y_val_w{window_size}.npy',   y_val)
    np.save(f'X_test_w{window_size}.npy',  X_test)
    np.save(f'y_test_w{window_size}.npy',  y_test)

print("\nAll 18 files saved.")


--- Window size 24 ---
Train: (48576, 24, 3) (48576,)
Val:   (11968, 24, 3) (11968,)
Test:  (11264, 24, 3) (11264,)

--- Window size 32 ---
Train: (47150, 32, 3) (47150,)
Val:   (11968, 32, 3) (11968,)
Test:  (11088, 32, 3) (11088,)

--- Window size 48 ---
Train: (43322, 48, 3) (43322,)
Val:   (11968, 48, 3) (11968,)
Test:  (10736, 48, 3) (10736,)

All 18 files saved.


In [16]:
from google.colab import files
for window_size in [24, 32, 48]:
    for split in ['train', 'val', 'test']:
        files.download(f'X_{split}_w{window_size}.npy')
        files.download(f'y_{split}_w{window_size}.npy')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# Confirm shapes make sense: (num_windows, window_size, 3 features)
for window_size in [24, 32, 48]:
    X_test = np.load(f'X_test_w{window_size}.npy')
    print(f"window={window_size} -> X_test shape:", X_test.shape)

window=24 -> X_test shape: (11264, 24, 3)
window=32 -> X_test shape: (11088, 32, 3)
window=48 -> X_test shape: (10736, 48, 3)
